In [ ]:
# !pip install pandas

In [ ]:
# #CLASSIFICAÇÃO FORÇADA DE ITENS NO BANCO DE DADOS

# import sqlite3
# import pandas as pd
# from pathlib import Path

# # --- CONFIGURAÇÃO ---
# # Copie EXATAMENTE como aparece na tela travada (parte do nome)
# TERMO_TRAVADO = "DROGRARIA SAO BRASILIA BR (Total 2x)" 
# # A categoria que deveria ter entrado
# CATEGORIA_FINAL = "Remédios"  

# db_path = Path(r"G:\Meu Drive\4. Registros\Glaydson\Orçamento\db\finance_abs.db")
# conn = sqlite3.connect(db_path)
# cursor = conn.cursor()

# print(f"🔍 Procurando fantasma: '{TERMO_TRAVADO}'...")

# # 1. Encontra o item problemático (pendente)
# df_ghost = pd.read_sql_query(
#     f"SELECT * FROM transactions WHERE description LIKE '%{TERMO_TRAVADO}%' AND category IS NULL", 
#     conn
# )

# if not df_ghost.empty:
#     display(df_ghost[['date', 'description', 'amount', 'category']])
    
#     confirm = input("Este é o item travado? (S/N): ").upper()
    
#     if confirm == 'S':
#         # Força a atualização
#         cursor.execute(f"""
#             UPDATE transactions 
#             SET category = '{CATEGORIA_FINAL}', is_manual = 1 
#             WHERE description LIKE '%{TERMO_TRAVADO}%' AND category IS NULL
#         """)
#         conn.commit()
#         print(f"✅ Item destravado! Foi classificado como '{CATEGORIA_FINAL}'.")
#     else:
#         print("Operação cancelada.")
# else:
#     print("Nenhum item pendente encontrado com esse nome. Pode ser que ele já esteja classificado, mas o Cache do Streamlit esteja enganando.")

# conn.close()

In [ ]:
# # CARREGAMENTO INICIAL
# import sqlite3
# import pandas as pd
# from pathlib import Path

# # Caminho do seu banco de dados (confirme se é este mesmo)
# db_path = Path(r"G:\Meu Drive\4. Registros\Glaydson\Orçamento\db\finance_abs.db")

# if not db_path.exists():
#     print(f"❌ Erro: Banco de dados não encontrado em: {db_path}")
#     print("Verifique se o Google Drive está conectado ou ajuste o caminho.")
# else:
#     print(f"✅ Banco encontrado: {db_path}")
#     conn = sqlite3.connect(db_path)

In [ ]:
# # RETIRAR O DADO DE DATA E HORÁRIO DE PIX

# import sqlite3
# import re
# import time
# from pathlib import Path

# # --- CONFIGURAÇÃO ---
# db_path = Path(r"G:\Meu Drive\4. Registros\Glaydson\Orçamento\db\finance_abs.db")
# TERMO_BUSCA = "Pix - Agendamento" 

# print(f"🔌 Conectando em: {db_path}")

# # Tenta conectar. Se estiver travado pelo Streamlit, vai avisar.
# try:
#     conn = sqlite3.connect(db_path, timeout=10) # Timeout de 10s
#     cursor = conn.cursor()
# except sqlite3.OperationalError:
#     print("❌ ERRO: O banco de dados está travado!")
#     print("⚠️ SOLUÇÃO: Feche a aba do Streamlit (pare o servidor) antes de rodar este script.")
#     raise

# # 1. Busca OTIMIZADA (Traz tudo para a memória de uma vez)
# print("🔍 Buscando transações...")
# start_fetch = time.time()
# rows = cursor.execute(
#     "SELECT hash_id, description FROM transactions WHERE description LIKE ?", 
#     (f'%{TERMO_BUSCA}%',)
# ).fetchall()
# print(f"📊 Encontrados: {len(rows)} registros (em {time.time() - start_fetch:.2f}s)")

# if len(rows) == 0:
#     print("✅ Nada a fazer. Banco limpo.")
#     conn.close()
# else:
#     # 2. Preparação das Ferramentas (Compila Regex para performance máxima)
#     # Padrão: data/data hora:hora (ex: 20/05 05:33)
#     regex_data = re.compile(r"\s*\d{2}/\d{2}\s+\d{2}:\d{2}\s*")
#     regex_hifen = re.compile(r"\s+-\s+-") # Remove duplo hífen gerado
#     regex_espacos = re.compile(r"\s+")    # Remove espaços duplos
    
#     updates = []
    
#     print(f"🚀 Iniciando análise de {len(rows)} itens...")
    
#     # 3. Processamento com Feedback Visual
#     for i, (hash_id, desc_original) in enumerate(rows):
        
#         # Mostra progresso a cada 1000 itens para você não achar que travou
#         if i > 0 and i % 1000 == 0:
#             print(f"   ... processados {i} itens ...")
            
#         # Aplica as limpezas
#         nova_desc = regex_data.sub(" ", desc_original) # Remove data/hora
#         nova_desc = regex_hifen.sub(" -", nova_desc)   # Corrige hifens
#         nova_desc = regex_espacos.sub(" ", nova_desc).strip() # Limpa espaços
        
#         if nova_desc != desc_original:
#             updates.append((nova_desc, hash_id))

#     print(f"🏁 Análise finalizada! {len(updates)} itens precisam ser corrigidos.")
    
#     # 4. Confirmação e Gravação em Lotes
#     if len(updates) > 0:
#         print("\n--- AMOSTRA DA MUDANÇA ---")
#         print(f"   DE: '{rows[0][1]}'")
#         print(f" PARA: '{updates[0][0]}'") # Mostra como ficou o primeiro item
#         print("--------------------------")
        
#         confirm = input("\n⚠️ Digite S para aplicar as correções agora: ").upper()
        
#         if confirm == 'S':
#             print("💾 Gravando no banco em lotes (Batch Update)...")
            
#             # Divide em lotes de 1000 para não travar
#             batch_size = 1000
#             total_batches = (len(updates) // batch_size) + 1
            
#             for i in range(0, len(updates), batch_size):
#                 batch = updates[i : i + batch_size]
#                 cursor.executemany(
#                     "UPDATE transactions SET description = ? WHERE hash_id = ?",
#                     batch
#                 )
#                 conn.commit() # Salva cada lote imediatamente
#                 print(f"   ✅ Lote {i//batch_size + 1}/{total_batches} salvo.")
                
#             print("\n🎉 SUCESSO! Todas as descrições foram limpas.")
#         else:
#             print("🚫 Operação cancelada. Nada foi alterado.")
#     else:
#         print("✅ Todos os itens encontrados já estão no padrão correto.")

#     conn.close()

In [ ]:
# ### EXCLUIR TODOS OS LANÇAMENTOS DO BANCO DE DADOS ###

# import sqlite3
# from pathlib import Path

# # --- CONFIGURAÇÃO ---
# db_path = Path(r"G:\Meu Drive\4. Registros\Glaydson\Orçamento\db\finance_abs.db")

# print(f"🔌 Conectando em: {db_path}")

# try:
#     conn = sqlite3.connect(db_path)
#     cursor = conn.cursor()

#     # 1. Verifica antes
#     qtd_antes = cursor.execute("SELECT count(*) FROM transactions").fetchone()[0]
#     print(f"📊 Lançamentos ANTES: {qtd_antes}")

#     if qtd_antes > 0:
#         # 2. AÇÃO: Apagar Tudo
#         print("🗑️ Apagando transações...")
#         cursor.execute("DELETE FROM transactions")
        
#         # 3. SALVAR IMEDIATAMENTE (O Pulo do Gato)
#         conn.commit()
#         print("💾 Alterações salvas com sucesso!")
        
#         # 4. Prova Real
#         qtd_depois = cursor.execute("SELECT count(*) FROM transactions").fetchone()[0]
#         print(f"📊 Lançamentos DEPOIS: {qtd_depois}")
        
#         if qtd_depois == 0:
#             print("\n✅ SUCESSO! Banco limpo. Pode reimportar.")
#         else:
#             print(f"\n❌ ERRO: Ainda restam {qtd_depois} registros.")
            
#     else:
#         print("✅ O banco já estava vazio.")

#     # Verifica se as regras sobreviveram (Segurança)
#     try:
#         qtd_regras = cursor.execute("SELECT count(*) FROM classification_rules").fetchone()[0]
#         print(f"🧠 Regras de Classificação mantidas: {qtd_regras}")
#     except:
#         print("⚠️ Tabela de regras não encontrada.")

# except sqlite3.OperationalError as e:
#     print(f"\n❌ ERRO DE ARQUIVO TRAVADO: {e}")
#     print("SOLUÇÃO: Feche o navegador do Streamlit e tente de novo.")

# finally:
#     if 'conn' in locals():
#         conn.close()

In [ ]:
# ### CONSULTAR CONTEÚDO DO BANCO DE DADOS ###
# import sqlite3
# import pandas as pd
# from pathlib import Path

# # Configuração do Caminho (O mesmo que configuramos no sistema)
# db_path = Path(r"G:\Meu Drive\4. Registros\Glaydson\Orçamento\db\finance_abs.db")

# if not db_path.exists():
#     print(f"❌ Banco não encontrado em: {db_path}")
#     print("Verifique se o Google Drive está conectado.")
# else:
#     conn = sqlite3.connect(db_path)
#     print(f"✅ Conectado com sucesso!")

#     # 1. Visão Geral (Últimos registros inseridos)
#     print("\n--- 🔍 Últimas 10 Transações Inseridas ---")
#     query_recent = "SELECT date, description, amount, source, category FROM transactions ORDER BY date DESC LIMIT 10"
#     df_recent = pd.read_sql_query(query_recent, conn)
#     display(df_recent)

In [ ]:
# #CORRIGE ERROS DE DIGITAÇÃO

# import sqlite3
# import pandas as pd
# from pathlib import Path

# # --- CONFIGURAÇÃO MANUAL (EDITE AQUI) ---
# DE_ERRADO = "Academia"     # <--- O nome que você quer eliminar
# PARA_CERTO = "Academias"   # <--- O nome correto que deve ficar
# # ----------------------------------------

# db_path = Path(r"G:\Meu Drive\4. Registros\Glaydson\Orçamento\db\finance_abs.db")
# conn = sqlite3.connect(db_path)
# cursor = conn.cursor()

# print(f"🔧 MODO DIRETO: Trocando '{DE_ERRADO}' por '{PARA_CERTO}'\n")

# # 1. Diagnóstico
# df_trans = pd.read_sql_query("SELECT * FROM transactions WHERE category = ?", conn, params=(DE_ERRADO,))
# df_rules = pd.read_sql_query("SELECT * FROM classification_rules WHERE target_category = ?", conn, params=(DE_ERRADO,))

# count_t = len(df_trans)
# count_r = len(df_rules)

# if count_t == 0 and count_r == 0:
#     print(f"⚠️ Nada encontrado com o nome '{DE_ERRADO}'. Verifique maiúsculas/minúsculas.")
# else:
#     print(f"📊 Encontrados: {count_t} transações e {count_r} regras.")
    
#     # 2. Execução Direta
#     # Atualiza transações
#     if count_t > 0:
#         cursor.execute("UPDATE transactions SET category = ? WHERE category = ?", (PARA_CERTO, DE_ERRADO))
#         print(f"✅ {cursor.rowcount} transações corrigidas.")
        
#     # Atualiza regras
#     if count_r > 0:
#         cursor.execute("UPDATE classification_rules SET target_category = ? WHERE target_category = ?", (PARA_CERTO, DE_ERRADO))
#         print(f"✅ {cursor.rowcount} regras atualizadas.")
        
#     conn.commit()
#     print("\n🚀 FEITO! Correção aplicada com sucesso.")

# conn.close()

In [ ]:
# import sqlite3
# import pandas as pd
# import re
# from src.database.connection import db_instance

# print("🔌 Conectando ao banco...")
# conn = db_instance.get_connection()

# # 1. Busca todas as transações
# df = pd.read_sql_query("SELECT hash_id, description, category FROM transactions", conn)

# updates = []
# # Regex para capturar "02/10", "05/12", etc.
# regex_pattern = re.compile(r'(\d{1,2})\s*/\s*(\d{1,2})')

# print(f"🔍 Analisando {len(df)} transações em busca de parcelas soltas...")

# for _, row in df.iterrows():
#     desc = row['description']
#     current_cat = row['category']
    
#     # Pula o que já está resolvido
#     if current_cat == "⛔ IGNORADO" or "Compra Parcelada" in str(desc):
#         continue

#     match = regex_pattern.search(str(desc))
#     if match:
#         try:
#             curr, total = map(int, match.groups())
            
#             # REGRA DE OURO: Se a parcela for maior que 1 (ex: 02/10, 03/05), IGNORA.
#             # Assumimos que a parcela 01 carrega o valor total (Regra de Competência).
#             if curr > 1:
#                 updates.append(row['hash_id'])
#         except:
#             pass

# if updates:
#     print(f"🧹 Faxina iniciada: Limpando {len(updates)} itens...")
    
#     cursor = conn.cursor()
#     # Atualização em lote (muito rápido)
#     placeholders = ', '.join(['?'] * len(updates))
#     sql = f"UPDATE transactions SET category = '⛔ IGNORADO', is_manual = 1 WHERE hash_id IN ({placeholders})"
    
#     cursor.execute(sql, updates)
#     conn.commit()
#     print(f"✅ SUCESSO! {cursor.rowcount} parcelas intermediárias foram marcadas como IGNORADO.")
# else:
#     print("✅ Nenhuma parcela solta encontrada. O banco está limpo!")

# conn.close()

INFO:src.database.connection:Conectado ao Banco Principal: G:\Meu Drive\4. Registros\Glaydson\Orçamento\db\finance_abs.db


🔌 Conectando ao banco...
🔍 Analisando 6284 transações em busca de parcelas soltas...
🧹 Faxina iniciada: Limpando 90 itens...
✅ SUCESSO! 90 parcelas intermediárias foram marcadas como IGNORADO.


In [1]:
### AJUSTE PONTUAL NO BANCO DE DADOS
# Arquivo: update_chat_schema.py
import sqlite3
from src.database.connection import db_instance

def create_chat_table():
    print("🧠 Criando lobo temporal para a IA...")
    conn = db_instance.get_connection()
    cursor = conn.cursor()
    
    try:
        # Tabela para guardar o histórico do chat por categoria
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS ai_chat_logs (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                category TEXT NOT NULL,
                role TEXT NOT NULL, -- 'user' ou 'model'
                message TEXT NOT NULL,
                timestamp DATETIME DEFAULT CURRENT_TIMESTAMP
            )
        """)
        conn.commit()
        print("✅ Memória da IA criada com sucesso!")
    except Exception as e:
        print(f"❌ Erro: {e}")
    finally:
        conn.close()

if __name__ == "__main__":
    create_chat_table()

INFO:src.database.connection:Conectado ao Banco Principal: G:\Meu Drive\4. Registros\Glaydson\Orçamento\db\finance_abs.db


🧠 Criando lobo temporal para a IA...
✅ Memória da IA criada com sucesso!


In [1]:
# teste_dados.py
from src.database.connection import db_instance
import pandas as pd

def testar_dados(category):
    conn = db_instance.get_connection()
    print(f"🔍 Buscando dados para categoria: '{category}'")
    
    # 1. Verifica se a categoria existe exatamente assim
    df_cat = pd.read_sql_query("SELECT DISTINCT category FROM transactions WHERE category LIKE ?", conn, params=(f"%{category}%",))
    print("Categorias encontradas no banco:", df_cat['category'].tolist())

    # 2. Roda a query do gráfico
    query = """
        SELECT strftime('%Y', date) as ano, strftime('%m', date) as mes, SUM(ABS(amount)) as valor
        FROM transactions 
        WHERE category = ? 
        AND strftime('%Y', date) IN ('2024', '2025', '2026') 
        AND amount < 0
        GROUP BY ano, mes
    """
    df = pd.read_sql_query(query, conn, params=(category,))
    
    print("\n--- Resultado da Query do Gráfico ---")
    if df.empty:
        print("❌ NENHUM DADO ENCONTRADO! O gráfico ficará vazio.")
    else:
        print(df)
        print("✅ Dados encontrados! O problema é no JavaScript.")

    conn.close()

if __name__ == "__main__":
    testar_dados("Compras Genéricas")

INFO:src.database.connection:Conectado ao Banco Principal: G:\Meu Drive\4. Registros\Glaydson\Orçamento\db\finance_abs.db


🔍 Buscando dados para categoria: 'Compras Genéricas'
Categorias encontradas no banco: ['Compras Genéricas']

--- Resultado da Query do Gráfico ---
     ano mes    valor
0   2024  01  1815.26
1   2024  02  4761.84
2   2024  03  7150.53
3   2024  04  3130.43
4   2024  05  2688.99
5   2024  06  4590.15
6   2024  07  4313.74
7   2024  08  3995.42
8   2024  09  9311.12
9   2024  10  5580.84
10  2024  11  2655.56
11  2024  12  5888.71
12  2025  01  4525.65
13  2025  02  2929.66
14  2025  03  4897.71
15  2025  04  6750.05
16  2025  05  3394.38
17  2025  06  1495.50
18  2025  07  4067.53
19  2025  08  4261.64
20  2025  09  8175.27
21  2025  10  5165.89
22  2025  11  4484.99
23  2025  12  5011.88
24  2026  01  6008.31
25  2026  02  3619.69
✅ Dados encontrados! O problema é no JavaScript.
